# Pipeline Evaluator
> Combine retrieval and RAG evaluation into one unified interface.

`PipelineEvaluator` orchestrates both retrieval-metric calculation
and LLM-based RAG evaluation, giving you a single object that can
score your entire pipeline end to end.

**Time:** ~5 minutes

## Setup

In [ ]:
from anchor.evaluation import (
    PipelineEvaluator,
    RetrievalMetricsCalculator,
    LLMRAGEvaluator,
)
from anchor.evaluation.models import RAGMetrics
from anchor.models import ContextItem, SourceType

## Prepare Components
Build the retrieval calculator and a mock LLM evaluator first,
then wire them into the `PipelineEvaluator`.

In [ ]:
calculator = RetrievalMetricsCalculator()

def mock_judge(query, answer, contexts, ground_truth=None):
    return RAGMetrics(
        faithfulness=0.9,
        relevancy=0.85,
        context_precision=0.8,
        context_recall=0.75,
    )

rag_evaluator = LLMRAGEvaluator(eval_fn=mock_judge)

print("Components ready:")
print(f"  RetrievalMetricsCalculator: {type(calculator).__name__}")
print(f"  LLMRAGEvaluator:            {type(rag_evaluator).__name__}")

## Create the Pipeline Evaluator

In [ ]:
pipeline_eval = PipelineEvaluator(
    retrieval_calculator=calculator,
    rag_evaluator=rag_evaluator,
)

print(f"PipelineEvaluator created: {type(pipeline_eval).__name__}")

## Build Mock Data
We need retrieved items and relevance labels for the retrieval
side, plus a query/answer pair for the RAG side.

In [ ]:
retrieved = [
    ContextItem(
        id=f"doc-{i}",
        content=f"Content about topic {i}",
        source=SourceType.RETRIEVAL,
        score=1.0 - i * 0.1,
        priority=5,
        token_count=10,
    )
    for i in range(5)
]
relevant_ids = ["doc-0", "doc-1", "doc-3"]

query = "What is context engineering?"
answer = "Context engineering is the systematic design of LLM context."
contexts = [item.content for item in retrieved[:3]]
ground_truth = "Context engineering designs the information surrounding LLM calls."

print(f"Retrieved items: {len(retrieved)}")
print(f"Relevant IDs:   {relevant_ids}")
print(f"Contexts sent:  {len(contexts)}")

## Evaluate Retrieval Only
`evaluate_retrieval()` delegates to the `RetrievalMetricsCalculator`.

In [ ]:
ret_metrics = pipeline_eval.evaluate_retrieval(
    retrieved=retrieved,
    relevant=relevant_ids,
    k=5,
)

print(f"Precision@5: {ret_metrics.precision:.3f}")
print(f"Recall@5:    {ret_metrics.recall:.3f}")
print(f"MRR:         {ret_metrics.mrr:.3f}")

## Evaluate RAG Only
`evaluate_rag()` delegates to the `LLMRAGEvaluator`.

In [ ]:
rag_metrics = pipeline_eval.evaluate_rag(
    query=query,
    answer=answer,
    contexts=contexts,
    ground_truth=ground_truth,
)

print(f"Faithfulness:      {rag_metrics.faithfulness:.2f}")
print(f"Relevancy:         {rag_metrics.relevancy:.2f}")
print(f"Context Precision: {rag_metrics.context_precision:.2f}")
print(f"Context Recall:    {rag_metrics.context_recall:.2f}")

## Full Pipeline Evaluation
`evaluate()` runs both retrieval and RAG evaluation in one call.

In [ ]:
full_result = pipeline_eval.evaluate(
    retrieved=retrieved,
    relevant=relevant_ids,
    query=query,
    answer=answer,
    contexts=contexts,
    ground_truth=ground_truth,
    k=5,
)

print("Full pipeline evaluation result:")
print(f"  Type: {type(full_result).__name__}")
print(f"  Contains retrieval + RAG metrics")

## Key Takeaways

- `PipelineEvaluator` unifies retrieval and RAG evaluation.
- Use `evaluate_retrieval()` or `evaluate_rag()` independently,
  or `evaluate()` for end-to-end scoring.
- Swapping components (e.g., a real LLM judge) requires no
  changes to the evaluation workflow.